In [1]:
import pandas as pd


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\nikol\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\nikol\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\nikol\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\Users\nikol\anaconda3\Lib\site-pack

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\nikol\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\nikol\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\nikol\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\Users\nikol\anaconda3\Lib\site-pack

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



In [9]:
df_dr = pd.read_csv("drkameloen-clean.csv")
df_subtlex = pd.read_csv(
    "SUBTLEX_CH_131210_CE.utf8",
    sep="\t",
    encoding="utf-8"
)

print(df_dr.columns.tolist())
print(df_subtlex.columns.tolist())

['simplified', 'traditional', 'pinyin', 'meaning', 'frequency', 'hsk_level', 'pos', 'radical']
['Word', 'Length', 'Pinyin', 'Pinyin.Input', 'WCount', 'W.million', 'log10W', 'W-CD', 'W-CD%', 'log10CD', 'Dominant.PoS', 'Dominant.PoS.Freq', 'All.PoS', 'All.PoS.Freq', 'Eng.Tran.']


In [18]:
def parse_dr_pos(pos):
    if pd.isna(pos):
        return set()

    return {
        p.strip()
        for p in str(pos).split(",")
        if p.strip()
    }


def parse_subtlex_pos(pos):
    if pd.isna(pos):
        return set()

    return {
        p.strip()
        for p in str(pos).strip(".").split(".")
        if p.strip()
    }


def normalize_erhua(word):
    if pd.isna(word):
        return word

    word = str(word).strip()

    if len(word) > 1 and word.endswith("儿"):
        return word[:-1]

    return word

In [19]:
df_dr["simplified_normalized"] = (
    df_dr["simplified"]
    .apply(normalize_erhua)
)

df_subtlex["Word_normalized"] = (
    df_subtlex["Word"]
    .astype(str)
    .str.strip()
)

In [20]:
def get_best_subtlex_row(dr_row):

    word = dr_row["simplified_normalized"]

    candidates = df_subtlex[
        df_subtlex["Word_normalized"] == word
    ]

    if len(candidates) == 0:
        return None

    dr_pos = parse_dr_pos(dr_row["pos"])

    candidates = candidates.copy()

    candidates["overlap"] = candidates["All.PoS"].apply(
        lambda x: len(
            dr_pos &
            parse_subtlex_pos(x)
        )
    )

    candidates = candidates.sort_values(
        ["overlap", "WCount"],
        ascending=[False, False]
    )

    return candidates.iloc[0]

In [22]:
merged_rows = []

for _, dr_row in df_dr.iterrows():

    best_subtlex = get_best_subtlex_row(dr_row)

    row = dr_row.to_dict()

    if best_subtlex is not None:

        for col in df_subtlex.columns:

            row[f"subtlex_{col}"] = best_subtlex[col]

    merged_rows.append(row)

df_merged = pd.DataFrame(merged_rows)

In [23]:
#2Menit 29 Detik
print(df_dr.shape)

print(df_merged.shape)

df_merged.head(2)

(10969, 9)
(10969, 25)


,simplified,traditional,pinyin,meaning,frequency,hsk_level,pos,radical,simplified_normalized,subtlex_Word,...,subtlex_log10W,subtlex_W-CD,subtlex_W-CD%,subtlex_log10CD,subtlex_Dominant.PoS,subtlex_Dominant.PoS.Freq,subtlex_All.PoS,subtlex_All.PoS.Freq,subtlex_Eng.Tran.,subtlex_Word_normalized
0,阿拉伯语,阿拉伯語,Ā lā bó yǔ,Arabic (language),27202,7,nz,阝,阿拉伯语,阿拉伯语,...,1.7160,34.0,0.54,1.5315,nz,52.0,.nz.,.52.,Arabic (language),阿拉伯语
1,阿姨,阿姨,ā yí,childcare worker ; maternal aunt ; nursemaid ;...,4355,4,n,阝,阿姨,阿姨,...,2.8987,392.0,6.28,2.5933,n,792.0,.n.,.792.,maternal aunt/step-mother/childcare worker/nur...,阿姨


In [26]:
matched = df_merged["subtlex_WCount"].notna().sum()

print("Matched :", matched)
print("Missing :", (len(df_merged) - matched))
print("Total   :", len(df_merged))
print("Rate    :", matched / len(df_merged))

Matched : 10760
Missing : 209
Total   : 10969
Rate    : 0.9809463032181602


In [27]:
df_dr['hsk_level'].value_counts()

hsk_level
7    5606
6    1123
5    1059
4     972
3     953
2     750
1     506
Name: count, dtype: int64

In [28]:
df_merged['hsk_level'].value_counts()

hsk_level
7    5606
6    1123
5    1059
4     972
3     953
2     750
1     506
Name: count, dtype: int64

In [ ]:
df_merged = df_merged.rename(columns={
    "subtlex_Word": "subtlex_word",
    "subtlex_Length": "subtlex_length",

    "subtlex_Pinyin": "subtlex_pinyin",
    "subtlex_Pinyin.Input": "subtlex_pinyin_input",

    "subtlex_WCount": "subtlex_w_count",
    "subtlex_W.million": "subtlex_w_million",

    "subtlex_log10W": "subtlex_log10w",

    "subtlex_W-CD": "subtlex_w_cd",
    "subtlex_W-CD%": "subtlex_w_cd_percent",

    "subtlex_log10CD": "subtlex_log10cd",

    "subtlex_Dominant.PoS": "subtlex_dominant_pos",
    "subtlex_Dominant.PoS.Freq": "subtlex_dominant_pos_freq",

    "subtlex_All.PoS": "subtlex_all_pos",
    "subtlex_All.PoS.Freq": "subtlex_all_pos_freq",

    "subtlex_Eng.Tran.": "subtlex_eng_tran",

    "subtlex_Word_normalized": "subtlex_word_normalized",
})

In [ ]:
df_merged.columns.tolist

In [ ]:
df_merged.to_csv(
    "mandarin_word_dataset_final.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved: mandarin_word_dataset_final.csv")